In [1]:
import pandas as pd 
import numpy as np
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
import warnings
load_dotenv()
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore")

In [2]:
url = os.getenv('URL')
engine = create_engine(url)

## Data Extraction

The dataset was stored in a MySQL database and extracted into Python for cleaning and analysis.

It contains store-level menu data including pricing, calories, and run dates across two quarters.


In [3]:
df=pd.read_sql("SELECT * FROM bg_king;", engine)
df.sample(5)

,STORE_ID,CITY,STATE,ZIP,MENU_SECTION,MENU_ITEM_NAME,MENU_ITEM_DESCRIPTION,CALORIES,MENU_PRICE,RUNDATE
116106,11554,Puyallup,WA,98375,Breakfast Sandwiches,Egg & Cheese Biscuit,Fluffy eggs and melted American cheese on a wa...,,4.99,2026-01-20
33546,13165,Houston,TX,77039,Condiments,Zesty Dipping Sauce,,150,0.25,2025-11-01
54457,11554,Puyallup,WA,98375,Meals,Whopper with Bacon Meal Medium,"America's Favorite Burger*, The Whopper Sandwi...",1470,13.18,2025-11-30
58779,11829,Worthington,MN,56187,Breakfast Sandwiches,Double Mix n' Match,,,6.00,2026-01-14
57492,11554,Puyallup,WA,98375,Drinks & Coffee,Pure LifeÂ® Purified Water,,,0.50,2026-01-08


## Initial Data Exploration

Before performing transformations, the dataset was explored to understand its structure, detect missing values, and identify potential data quality issues.


In [4]:
print("Total Rows : ",df.shape[0])
print("Total Columns : ",df.shape[1])

Total Rows :  119210
Total Columns :  10


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119210 entries, 0 to 119209
Data columns (total 10 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   STORE_ID               119210 non-null  int64  
 1   CITY                   119210 non-null  object 
 2   STATE                  119210 non-null  object 
 3   ZIP                    119210 non-null  int64  
 4   MENU_SECTION           119210 non-null  object 
 5   MENU_ITEM_NAME         119210 non-null  object 
 6   MENU_ITEM_DESCRIPTION  119210 non-null  object 
 7   CALORIES               119210 non-null  object 
 8   MENU_PRICE             119210 non-null  float64
 9   RUNDATE                119210 non-null  object 
dtypes: float64(1), int64(2), object(7)
memory usage: 9.1+ MB


#### Converting empty strings into null values of all column

In [6]:
for col in df:
   df[col] = df[col] \
    .replace('', np.nan)    

### Date Formatting

The RUNDATE column was converted to datetime format to enable time-based analysis such as quarter-over-quarter comparisons.


In [7]:
df['RUNDATE'] = pd.to_datetime(df['RUNDATE'])

In [8]:
df.sample(5)

,STORE_ID,CITY,STATE,ZIP,MENU_SECTION,MENU_ITEM_NAME,MENU_ITEM_DESCRIPTION,CALORIES,MENU_PRICE,RUNDATE
114821,7560,Sturtevant,WI,53177,Drinks & Coffee,Pure LifeÂ® Purified Water,A cool and refreshing way to wash down your si...,NaN,-0.80,2026-01-20
23430,28727,Seguin,TX,78155,Drinks & Coffee,Powerade Zero,NaN,15,2.69,2026-01-07
26587,28727,Seguin,TX,78155,Drinks & Coffee,Pure LifeÂ® Purified Water,A cool and refreshing way to wash down your si...,NaN,1.76,2026-01-20
42511,13165,Houston,TX,77039,Drinks & Coffee,Medium Unsweetened Iced Tea,Brewed fresh daily.,NaN,0.60,2025-12-11
19990,2010,E Stroudsburg,PA,18301,Flame Grilled Burgers,Whopper Jr.Â® with Bacon Meal Medium,A flame-grilled beef patty with juicy tomatoes...,750,8.69,2026-01-06


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119210 entries, 0 to 119209
Data columns (total 10 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   STORE_ID               119210 non-null  int64         
 1   CITY                   119210 non-null  object        
 2   STATE                  119210 non-null  object        
 3   ZIP                    119210 non-null  int64         
 4   MENU_SECTION           119210 non-null  object        
 5   MENU_ITEM_NAME         119210 non-null  object        
 6   MENU_ITEM_DESCRIPTION  87322 non-null   object        
 7   CALORIES               72872 non-null   object        
 8   MENU_PRICE             119210 non-null  float64       
 9   RUNDATE                119210 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), int64(2), object(6)
memory usage: 9.1+ MB


### Categorical Data Review

Unique values across location and menu-related columns were examined to understand geographic coverage and menu structure.


In [10]:
df['STORE_ID'].unique()

array([13165,  7560, 11554, 11829, 12190, 28727,  3335,   266, 28777,
        2010], dtype=int64)

In [11]:
df['CITY'].unique()

array(['Houston', 'Sturtevant', 'Puyallup', 'Worthington', 'West Bend',
       'Seguin', 'Tucson', 'El Paso', 'E Stroudsburg'], dtype=object)

In [12]:
df['STATE'].unique()

array(['TX', 'WI', 'WA', 'MN', 'AZ', 'PA'], dtype=object)

In [13]:
df['ZIP'].unique()

array([77039, 53177, 98375, 56187, 53095, 78155, 85730, 77055, 79928,
       18301], dtype=int64)

In [14]:
df['MENU_SECTION'].unique()

array(['Condiments', 'Breakfast Digital Exclusives', 'Drinks & Coffee',
       'Breakfast Meals', 'Breakfast Sides', 'Chicken & Fish', 'Sides',
       'Meals', 'Monster Menu', 'Sweets', 'King Jr. Kids Meals',
       'Limited Time Only', 'Burgers for Breakfast',
       'Flame Grilled Burgers', 'Breakfast Sandwiches', 'Burritos',
       'NEEDED FOR PAR', 'Digital Exclusives', 'Whopper by You'],
      dtype=object)

In [15]:
for col in df.select_dtypes('object'):
    print(col, df[col].nunique())

CITY 9
STATE 6
MENU_SECTION 19
MENU_ITEM_NAME 669
MENU_ITEM_DESCRIPTION 195
CALORIES 203


## Data Cleaning & Preparation

### Price Correction

Negative price values were identified and corrected to ensure accurate financial analysis.


In [16]:
df['MENU_PRICE'].sort_values()

60765    -1.59
64659    -1.59
77557    -1.59
28773    -1.59
65112    -1.59
         ...  
17858    31.99
91427    32.99
91437    32.99
80751    32.99
80268    32.99
Name: MENU_PRICE, Length: 119210, dtype: float64

In [17]:
df['MENU_PRICE'] = abs(df['MENU_PRICE'])

In [18]:
df['MENU_PRICE'].sort_values()

70538     0.01
13890     0.01
54350     0.01
37800     0.01
47262     0.01
         ...  
17858    31.99
80268    32.99
91437    32.99
91427    32.99
80751    32.99
Name: MENU_PRICE, Length: 119210, dtype: float64

In [19]:
df['MENU_ITEM_NAME'].unique()

array(['Zesty Dipping Sauce', 'Get Going Trio', 'Chocolate OreoÂ® Shake',
       '5 Pc. French Toast Sticks with Syrup Meal Large',
       'Medium French Fries', 'BBQ Dipping Sauce', '1 Pc. Taco',
       '12 Pc. Chicken Fries', '12 Pc. Chicken Fries Meal Large',
       '12 Pc. Chicken Fries Meal Medium',
       '12 Pc. Chicken Fries Meal Small', '12 Pc. Mozzarella Fries',
       '12 Pc. Mozzarella Fries Meal Large',
       '12 Pc. Mozzarella Fries Meal Medium',
       '12 Pc. Mozzarella Fries Meal Small', '16 Pc. Chicken Nuggets',
       '16 Pc. Chicken Nuggets Meal Large',
       '16 Pc. Chicken Nuggets Meal Medium',
       '16 Pc. Chicken Nuggets Meal Small', '2 Chocolate Chip Cookies',
       '3 Pc. French Toast Sticks', '4 Pc. Cheesy Tots',
       '4 Pc. Chicken Fries', '4 Pc. Chicken Nuggets',
       '4 Pc. Chicken Nuggets King Jr. Meal', '4 Pc. Cini Minis',
       '4 Pc. JalapeÃ±o Cheddar Bites', '4 Pc. Mozzarella Fries',
       '5 Pc. French Toast Sticks',
       '5 Pc. French T

## Time Feature Engineering

New temporal features such as month and quarter were derived from the run date to support aggregated reporting and seasonal analysis.

In [20]:
df['RUNDATE'].sort_values().unique()

<DatetimeArray>
['2025-10-01 00:00:00', '2025-11-01 00:00:00', '2025-11-04 00:00:00',
 '2025-11-14 00:00:00', '2025-11-30 00:00:00', '2025-12-01 00:00:00',
 '2025-12-05 00:00:00', '2025-12-11 00:00:00', '2026-01-01 00:00:00',
 '2026-01-06 00:00:00', '2026-01-07 00:00:00', '2026-01-08 00:00:00',
 '2026-01-14 00:00:00', '2026-01-20 00:00:00', '2026-01-21 00:00:00',
 '2026-01-31 00:00:00', '2026-02-01 00:00:00', '2026-02-02 00:00:00']
Length: 18, dtype: datetime64[ns]

In [21]:
df['QUARTER'] = df['RUNDATE'].dt.to_period('Q')

In [22]:
df['QUARTER'] = df['QUARTER'].astype(str)

In [23]:
df['Month'] = df['RUNDATE'].dt.month_name()

In [24]:
df = df.drop(columns='RUNDATE')

### Handling Missing Nutritional Values

Missing calorie values were investigated to determine whether they resulted from data entry gaps or product variations.


In [25]:
df[df['CALORIES'].isna()].shape

(46338, 11)

In [26]:
problem_items = df.groupby('MENU_ITEM_NAME')['CALORIES'] \
    .agg(['count','size'])

problem_items[problem_items['count'] != problem_items['size']]

,count,size
MENU_ITEM_NAME,,
12 Pc. Hidden ValleyÂ® Ranch Chicken Fries Meal Large,3,4
16 Pc. Chicken Nuggets Meal Large,136,139
16 Pc. Chicken Nuggets Meal Medium,138,139
2 Chocolate Chip Cookies,202,302
4 Pc. Chicken Nuggets King Jr. Meal,330,531
...,...,...
Whopper,712,783
Whopper Jr Meals,79,130
Whopper Jr.,596,695


### Calorie Standardization

Calorie inconsistencies were traced to variations in menu item descriptions.  
Product-level mappings were applied to standardize zero-calorie beverages and ensure analytical accuracy.

In [27]:
df['CALORIES'] = df.groupby('MENU_ITEM_NAME')['CALORIES'] \
                   .transform(lambda x: x.ffill().bfill())

In [28]:
problem_items = df.groupby('MENU_ITEM_NAME')['CALORIES'] \
    .agg(['count','size'])

problem_items[problem_items['count'] != problem_items['size']]

,count,size
MENU_ITEM_NAME,,
Brewed Coffee,0,133
Coca-Cola Zero Sugar,0,130
Decaf Coffee,0,113
Diet Barq's,0,68
Diet Coke,0,130
Large BK CafÃ©,0,11
Large BK CafÃ© Decaf,0,9
Large Black Iced Coffee,0,170
Large Brewed Coffee,0,127


In [29]:
df['CALORIES'] = df['CALORIES'].fillna(0)

In [30]:
df[df['CALORIES'] == 0]

,STORE_ID,CITY,STATE,ZIP,MENU_SECTION,MENU_ITEM_NAME,MENU_ITEM_DESCRIPTION,CALORIES,MENU_PRICE,QUARTER,Month
708,13165,Houston,TX,77039,Drinks & Coffee,Large Black Iced Coffee,Our BKÂ® CafÃ© Iced Coffee starts with 100% Ar...,0,2.69,2025Q4,November
709,13165,Houston,TX,77039,Drinks & Coffee,Large Black Iced Coffee,Our BKÂ® CafÃ© Iced Coffee starts with 100% Ar...,0,0.15,2025Q4,November
710,13165,Houston,TX,77039,Drinks & Coffee,Large Black Iced Coffee,Our BKÂ® CafÃ© Iced Coffee starts with 100% Ar...,0,0.50,2025Q4,November
711,13165,Houston,TX,77039,Drinks & Coffee,Large Black Iced Coffee,Our BKÂ® CafÃ© Iced Coffee starts with 100% Ar...,0,0.10,2025Q4,November
712,7560,Sturtevant,WI,53177,Drinks & Coffee,Large Black Iced Coffee,Our BKÂ® CafÃ© Iced Coffee starts with 100% Ar...,0,2.00,2025Q4,November
...,...,...,...,...,...,...,...,...,...,...,...
118935,7560,Sturtevant,WI,53177,Drinks & Coffee,Small Coca-Cola Zero,"Â© 2018 The Coca-Cola Company. ""Coca-Cola"""" is...",0,2.49,2026Q1,February
118936,7560,Sturtevant,WI,53177,Drinks & Coffee,Small Decaf Coffee,Our blend is made with 100% Arabica beans and ...,0,1.00,2026Q1,February
118937,7560,Sturtevant,WI,53177,Drinks & Coffee,Small Diet Coke,Try a crisp and refreshing no-calorie Diet Cok...,0,2.49,2026Q1,February
119102,7560,Sturtevant,WI,53177,Drinks & Coffee,Value Coca-Cola Zero,"Â© 2018 The Coca-Cola Company. ""Coca-Cola"""" is...",0,1.89,2026Q1,February


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119210 entries, 0 to 119209
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   STORE_ID               119210 non-null  int64  
 1   CITY                   119210 non-null  object 
 2   STATE                  119210 non-null  object 
 3   ZIP                    119210 non-null  int64  
 4   MENU_SECTION           119210 non-null  object 
 5   MENU_ITEM_NAME         119210 non-null  object 
 6   MENU_ITEM_DESCRIPTION  87322 non-null   object 
 7   CALORIES               119210 non-null  object 
 8   MENU_PRICE             119210 non-null  float64
 9   QUARTER                119210 non-null  object 
 10  Month                  119210 non-null  object 
dtypes: float64(1), int64(2), object(8)
memory usage: 10.0+ MB


### Data Type Corrections

The CALORIES column was converted to a numeric format to support aggregation and statistical analysis.


In [32]:
df['CALORIES'] = pd.to_numeric(df['CALORIES'], errors='coerce')

In [33]:
df.sample(5)

,STORE_ID,CITY,STATE,ZIP,MENU_SECTION,MENU_ITEM_NAME,MENU_ITEM_DESCRIPTION,CALORIES,MENU_PRICE,QUARTER,Month
44958,7560,Sturtevant,WI,53177,Flame Grilled Burgers,Double Cheeseburger,Two flame-grilled patties. 1/4 lb* of 100% bee...,400,3.49,2026Q1,January
44337,11554,Puyallup,WA,98375,Chicken & Fish,Spicy Royal Crispy Chicken,NaN,760,6.99,2026Q1,January
63224,12190,West Bend,WI,53095,Sweets,2 Chocolate Chip Cookies,Our delicious Chocolate Chip cookies are loade...,320,1.00,2026Q1,January
10710,3335,Tucson,AZ,85730,Meals,Texas Double WhopperÂ® Meal Small,Two Â¼ lb.* flame-grilled beef patties topped ...,1600,12.29,2026Q1,January
96053,12190,West Bend,WI,53095,Sides,Onion Rings,NaN,360,3.29,2026Q1,January


In [34]:
df[df['MENU_ITEM_DESCRIPTION'].isna()].shape

(31888, 11)

In [35]:
problem_desc = df.groupby(['MENU_ITEM_NAME'])['MENU_ITEM_DESCRIPTION'] \
    .agg(['count','size'])

problem_desc[problem_desc['count'] != problem_desc['size']]

,count,size
MENU_ITEM_NAME,,
12 Pc. Chicken Fries,861,1022
12 Pc. Chicken Fries Meal,0,2
12 Pc. Mozzarella Fries,345,393
16 Pc. Chicken Nuggets,673,761
2 Chocolate Chip Cookies,223,302
...,...,...
Whopper Jr. Meal,0,2
Whopper Meal,0,2
Whopper Meals,0,130


In [36]:
df['MENU_ITEM_DESCRIPTION'] = df.groupby(['MENU_ITEM_NAME'])['MENU_ITEM_DESCRIPTION'] \
                   .transform(lambda x: x.ffill().bfill())

In [37]:
problem_desc = df.groupby(['MENU_ITEM_NAME'])['MENU_ITEM_DESCRIPTION'] \
    .agg(['count','size'])

problem_desc[problem_desc['count'] != problem_desc['size']]

,count,size
MENU_ITEM_NAME,,
12 Pc. Chicken Fries Meal,0,2
4 Pc. Nuggets,0,18
8 Pc. Chicken Fries Meal,0,2
BBQ Dipping Sauce,0,583
Bacon & Swiss Royal Crispy Chicken Meals,0,128
...,...,...
Whopper Jr Meals,0,130
Whopper Jr. Meal,0,2
Whopper Meal,0,2


In [38]:
df[df['MENU_ITEM_DESCRIPTION'].isna()].shape

(13055, 11)

## Duplicate Record Handling

Duplicate rows were identified during validation and removed to prevent inflated counts and misleading aggregations.

In [39]:
df.duplicated().sum()

83942

In [40]:
print("Before:", df.shape)

Before: (119210, 11)


In [41]:
df_clean = df.drop_duplicates().copy()

In [42]:
print("After:", df_clean.shape)

After: (35268, 11)


In [43]:
df_clean.duplicated().sum()

0

In [44]:
df_clean.groupby('Month')['MENU_ITEM_NAME'].size()


Month
December    7864
February    6904
January     8171
November    8098
October     4231
Name: MENU_ITEM_NAME, dtype: int64

In [45]:
df_clean.groupby('Month')['MENU_ITEM_NAME'].nunique()

Month
December    540
February    526
January     577
November    518
October     493
Name: MENU_ITEM_NAME, dtype: int64

In [46]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 35268 entries, 0 to 119208
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   STORE_ID               35268 non-null  int64  
 1   CITY                   35268 non-null  object 
 2   STATE                  35268 non-null  object 
 3   ZIP                    35268 non-null  int64  
 4   MENU_SECTION           35268 non-null  object 
 5   MENU_ITEM_NAME         35268 non-null  object 
 6   MENU_ITEM_DESCRIPTION  33909 non-null  object 
 7   CALORIES               35268 non-null  int64  
 8   MENU_PRICE             35268 non-null  float64
 9   QUARTER                35268 non-null  object 
 10  Month                  35268 non-null  object 
dtypes: float64(1), int64(3), object(7)
memory usage: 3.2+ MB


## Clean Data Storage

After preprocessing, the cleaned dataset was written back to MySQL to support efficient querying and downstream visualization.


In [47]:
df_clean.to_sql(
    name='bg_king_clean',
    con=engine,
    if_exists='replace',  
    index=False
)

-1